# Notebook 05: Results, Confusion Matrices, and Discussion

This notebook generates the final expanded evaluation outputs: confusion matrices, classification reports, ROC and precision-recall curves, model comparison tables, and the HTML dashboard.

## Short Problem Statement

Existing deep learning-based **Intrusion Detection Systems (IDS)** often rely mainly on time-domain flow features and may miss hidden burst patterns, repeated traffic rhythms, and frequency-based attack signatures. **SentinelFlow** addresses this by using **Fast Fourier Transform (FFT)**-enhanced traffic profiling to improve deep learning-based intrusion detection on large-scale network traffic datasets.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'sentinelflow_utils.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from sentinelflow_utils import *
ensure_dirs(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
results_path = PROJECT_ROOT / 'outputs/metrics/04_model_results_expanded.csv'
pred_path = PROJECT_ROOT / 'outputs/predictions/04_all_model_predictions.csv'
if not results_path.exists():
    raise FileNotFoundError('Run Notebook 04 first. Missing model results.')
results_df = pd.read_csv(results_path)
predictions_df = pd.read_csv(pred_path) if pred_path.exists() else pd.DataFrame()
print('Results shape:', results_df.shape)
print('Predictions shape:', predictions_df.shape)
display(results_df.sort_values(['task','macro_f1'], ascending=[True, False]).head(20))

In [ ]:
# Load class distribution for dashboard.
class_dist_path = PROJECT_ROOT / 'outputs/class_distribution_attack.csv'
if class_dist_path.exists():
    class_dist = pd.read_csv(class_dist_path)
else:
    cleaned_path = PROJECT_ROOT / 'data/processed/sentinelflow_cleaned_netflow.parquet'
    if cleaned_path.exists():
        class_dist = class_distribution(pd.read_parquet(cleaned_path), 'target_attack')
    else:
        class_dist = pd.DataFrame(columns=['target_attack','count','percent'])
class_dist.head(20)

In [ ]:
# Generate confusion matrices for every model/task pair.
cm_dir = PROJECT_ROOT / 'reports/confusion_matrices'
cm_dir.mkdir(parents=True, exist_ok=True)
image_paths = []
if not predictions_df.empty:
    for _, pair in predictions_df[['feature_set','task','model']].drop_duplicates().iterrows():
        fs, task, model = pair['feature_set'], pair['task'], pair['model']
        sub = predictions_df[(predictions_df['feature_set'] == fs) & (predictions_df['task'] == task) & (predictions_df['model'] == model)].copy()
        if sub.empty:
            continue
        safe = f"{task}_{fs}_{model}".replace(' ', '_').replace('/', '_').replace('+','plus').replace(':','')
        if task == 'binary':
            labels = [0, 1]
            title = f"Binary Confusion Matrix: {model} ({fs})"
            p = save_confusion_matrix_plot(sub['y_true'].astype(int), sub['y_pred'].astype(int), labels, title, cm_dir / f'{safe}_confusion_matrix.png')
            image_paths.append(Path('confusion_matrices') / p.name)
        else:
            labels = sorted(set(sub['y_true'].astype(str)).union(set(sub['y_pred'].astype(str))))
            # Keep plot readable for many classes by using top labels that appear.
            if len(labels) <= 15:
                title = f"Multiclass Confusion Matrix: {model} ({fs})"
                p = save_confusion_matrix_plot(sub['y_true'].astype(str), sub['y_pred'].astype(str), labels, title, cm_dir / f'{safe}_confusion_matrix.png')
                image_paths.append(Path('confusion_matrices') / p.name)
print('Generated image count:', len(image_paths))
image_paths[:5]

In [ ]:
# Generate ROC and Precision-Recall curves for binary models.
curve_paths = save_roc_pr_curves(predictions_df[predictions_df['task'] == 'binary'].copy(), PROJECT_ROOT / 'reports/figures') if not predictions_df.empty else []
image_paths.extend([Path('figures') / p.name for p in curve_paths])
print('Generated curve count:', len(curve_paths))

In [ ]:
# Summaries by feature set and task.
summary_cols = ['feature_set','task','accuracy','balanced_accuracy','precision','recall','specificity','f1','macro_f1','weighted_f1','false_positive_rate','false_negative_rate','roc_auc','pr_auc']
summary_cols = [c for c in summary_cols if c in results_df.columns]
feature_summary = results_df.groupby(['feature_set','task'], dropna=False)[summary_cols[2:]].mean(numeric_only=True).reset_index()
feature_summary.to_csv(PROJECT_ROOT / 'outputs/metrics/05_feature_set_summary.csv', index=False)
display(feature_summary)

In [ ]:
# Build HTML dashboard.
summary = {
    'rows': 'See dataset summary',
    'features': 'Baseline + FFT features',
}
try:
    profile = json.loads((PROJECT_ROOT / 'outputs/database_profile.json').read_text(encoding='utf-8'))
    summary['rows'] = profile.get('clean_shape', ['N/A'])[0]
    summary['features'] = profile.get('clean_shape', ['N/A', 'N/A'])[1]
except Exception:
    pass
html = make_html_dashboard(results_df, class_dist, summary, image_paths=image_paths)
dashboard_path = PROJECT_ROOT / 'reports/sentinelflow_v3_results_dashboard.html'
dashboard_path.write_text(html, encoding='utf-8')
print('Saved dashboard:', dashboard_path)

In [ ]:
# Consultation-ready conclusion text.
valid = results_df.dropna(subset=['macro_f1']) if 'macro_f1' in results_df.columns else pd.DataFrame()
if len(valid):
    best = valid.sort_values('macro_f1', ascending=False).iloc[0]
    print('Best model:', best['model'])
    print('Feature set:', best['feature_set'])
    print('Task:', best['task'])
    print('Macro F1-score:', best['macro_f1'])
    print('
Interpretation:')
    print('SentinelFlow now produces expanded evaluation outputs, including confusion matrices, per-class reports, ROC-AUC, PR-AUC, false positive rate, and false negative rate. Final research claims should be made only after running on the full dataset, not only the bundled sample.')
else:
    print('No valid model results found. Rerun Notebook 04.')